In [7]:
from ase.io import read, write
initial_path = "/home/mehuldarak/athena/Case3VASP/Case3.cif"

atoms = read(initial_path)

symbols = atoms.get_chemical_symbols()
li_indices = [i for i, s in enumerate(symbols) if s == "Li"]

# pick source
i0 = li_indices[0]

# find nearest Li (target position)
min_dist = 1e9
i1 = None

for j in li_indices:
    if j == i0:
        continue
    d = atoms.get_distance(i0, j, mic=True)
    if d < min_dist:
        min_dist = d
        i1 = j

# create final
final = atoms.copy()

# move Li i0 → position of Li i1
final.positions[i0] = atoms.positions[i1]

write("final.cif", final)

print("Fixed final.cif created ✅")

Fixed final.cif created ✅


In [ ]:
from ase.io import read, write
from ase.mep import NEB
from ase.optimize import MDMin
from ase.constraints import FixAtoms
from mace.calculators import MACECalculator
import torch

# =========================
# CONFIG
# =========================
model_path = "/home/mehuldarak/athena/mace_fps_training/mace_fps_split17_compiled.model"
initial_path = "/home/mehuldarak/athena/Case3VASP/Case3.cif"

n_images = 9          # more images = smoother path
fmax_stage1 = 0.1
fmax_stage2 = 0.05

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD STRUCTURE
# =========================
initial = read(initial_path)

symbols = initial.get_chemical_symbols()
li_indices = [i for i, s in enumerate(symbols) if s == "Li"]

# pick source Li
i0 = li_indices[0]

# find nearest Li
min_dist = 1e9
i1 = None
for j in li_indices:
    if j == i0:
        continue
    d = initial.get_distance(i0, j, mic=True)
    if d < min_dist:
        min_dist = d
        i1 = j

print(f"Li hop: {i0} → {i1} (distance = {min_dist:.3f} Å)")

# =========================
# CREATE VACANCY STRUCTURES
# =========================
initial_vac = initial.copy()
final_vac   = initial.copy()

# remove Li at destination → vacancy
del initial_vac[i1]
del final_vac[i1]

# adjust index after deletion
if i1 < i0:
    i0 -= 1

# move Li into vacancy
final_vac.positions[i0] = initial.positions[i1]

# =========================
# SANITY CHECK
# =========================
assert len(initial_vac) == len(final_vac)

for i, (a, b) in enumerate(zip(initial_vac, final_vac)):
    if a.symbol != b.symbol:
        raise ValueError(f"Mismatch at {i}: {a.symbol} vs {b.symbol}")

print("Sanity check passed ✔")

# =========================
# OPTIONAL: CONSTRAINTS (freeze non-Li)
# =========================
mask = [atom.symbol != "Li" for atom in initial_vac]
constraint = FixAtoms(mask=mask)

# =========================
# CREATE IMAGES
# =========================
images = [initial_vac.copy()]

for _ in range(n_images - 2):
    images.append(initial_vac.copy())

images.append(final_vac.copy())

for img in images:
    img.set_constraint(constraint)

# =========================
# ATTACH CALCULATORS
# =========================
def attach_calculators(images):
    for img in images:
        img.calc = MACECalculator(
            model_paths=[model_path],
            device=device,
            default_dtype="float32"
        )

attach_calculators(images)

# =========================
# STAGE 1: RELAX BAND (no climb)
# =========================
print("\n--- Stage 1: Band relaxation ---")

neb = NEB(images, k=0.2, climb=False)
neb.interpolate(method="idpp")

optimizer = MDMin(neb, trajectory="neb_stage1.traj", logfile="neb_stage1.log")
optimizer.run(fmax=fmax_stage1)

# =========================
# STAGE 2: CLIMBING IMAGE
# =========================
print("\n--- Stage 2: Climbing image ---")

attach_calculators(images)  # reattach fresh calculators

neb = NEB(images, k=0.2, climb=True)

optimizer = MDMin(neb, trajectory="neb_stage2.traj", logfile="neb_stage2.log")
optimizer.run(fmax=fmax_stage2)

# =========================
# RESULTS
# =========================
energies = [img.get_potential_energy() for img in images]

E0 = energies[0]
barrier = max(energies) - E0

print("\n===== NEB RESULTS =====")
for i, e in enumerate(energies):
    print(f"Image {i}: {e:.6f} eV")

print(f"\nMigration barrier: {barrier:.6f} eV")

# =========================
# SAVE FINAL BAND
# =========================
write("neb_band.xyz", images)
print("\nSaved neb_band.xyz ✔")

Li hop: 0 → 9 (distance = 2.799 Å)
Sanity check passed ✔
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']

--- Stage 1: Band relaxation ---


In [8]:
from ase.io import read
from mace.calculators import MACECalculator
import torch

model_path = "/home/mehuldarak/athena/mace_fps_training/mace_fps_split17_compiled.model"
device = "cuda" if torch.cuda.is_available() else "cpu"

# load images correctly
images = read("neb_band.xyz", index=":")

print(f"Loaded {len(images)} images")

# attach calc
for img in images:
    img.calc = MACECalculator(
        model_paths=[model_path],
        device=device,
        default_dtype="float32"
    )

energies = [img.get_potential_energy() for img in images]

E0 = energies[0]
barrier = max(energies) - E0

print("\nEnergies:")
for i, e in enumerate(energies):
    print(f"Image {i}: {e:.6f} eV")

print(f"\n🔥 Migration barrier: {barrier:.6f} eV")

Loaded 9 images
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']

Energies:
Image 0: -202365.500000 eV
Image 1: -202388.796875 eV
Image 2: -202388.593750 eV
Image 3: -202388.281250 eV
Image 4: -202388.281250 eV
Image 5: -202387.765625 eV
Image 6: -202388.000000 eV
Image 7: -202388.687500 eV
Image 8: -202365.281250 eV

🔥 Migration barrier: 0.218750 eV


In [ ]:
from ase.io import read, write
from ase.mep import NEB
from ase.optimize import BFGS, MDMin
from ase.constraints import FixAtoms
from mace.calculators import MACECalculator
import torch

# =========================
# CONFIG
# =========================
model_path = "/home/mehuldarak/athena/mace_fps_training/mace_fps_split17_compiled.model"
initial_path = "/home/mehuldarak/athena/Case3VASP/Case3.cif"

n_images = 9
device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD STRUCTURE
# =========================
initial = read(initial_path)

symbols = initial.get_chemical_symbols()
li_indices = [i for i, s in enumerate(symbols) if s == "Li"]

# pick Li and nearest neighbor
i0 = li_indices[0]

min_dist = 1e9
i1 = None
for j in li_indices:
    if j == i0:
        continue
    d = initial.get_distance(i0, j, mic=True)
    if d < min_dist:
        min_dist = d
        i1 = j

print(f"Li hop: {i0} → {i1} (distance = {min_dist:.3f} Å)")

# =========================
# CREATE VACANCY STRUCTURES
# =========================
initial_vac = initial.copy()
final_vac   = initial.copy()

del initial_vac[i1]
del final_vac[i1]

if i1 < i0:
    i0 -= 1

final_vac.positions[i0] = initial.positions[i1]

# =========================
# CALCULATOR FUNCTION
# =========================
def get_calc():
    return MACECalculator(
        model_paths=[model_path],
        device=device,
        default_dtype="float32"
    )

# =========================
# RELAX ENDPOINTS (CRITICAL)
# =========================
print("\n--- Relaxing endpoints ---")

for name, atoms in [("Initial", initial_vac), ("Final", final_vac)]:
    atoms.calc = get_calc()
    opt = BFGS(atoms, logfile=f"{name.lower()}_relax.log")
    opt.run(fmax=0.02)
    print(f"{name} relaxed ✔")

# =========================
# OPTIONAL: CONSTRAINTS
# =========================
mask = [atom.symbol != "Li" for atom in initial_vac]
constraint = FixAtoms(mask=mask)

# =========================
# CREATE NEB IMAGES
# =========================
images = [initial_vac.copy()]

for _ in range(n_images - 2):
    images.append(initial_vac.copy())

images.append(final_vac.copy())

for img in images:
    img.set_constraint(constraint)
    img.calc = get_calc()



Li hop: 0 → 9 (distance = 2.799 Å)

--- Relaxing endpoints ---
Using head Default out of ['Default']
Initial relaxed ✔
Using head Default out of ['Default']
Final relaxed ✔
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']
Using head Default out of ['Default']

--- Stage 1: Band relaxation ---


RuntimeError: Constraints in image 1
affect the interpolation results.
Please specify if you want to
apply or ignore the constraints
during the interpolation
with the apply_constraint argument.

In [ ]:
# =========================
# STAGE 1: RELAX BAND
# =========================
print("\n--- Stage 1: Band relaxation ---")

neb = NEB(images, k=0.2, climb=False)
neb.interpolate(method="idpp")

opt = MDMin(neb, trajectory="neb_stage1.traj", logfile="neb_stage1.log")
opt.run(fmax=0.1)

# =========================
# STAGE 2: CLIMBING IMAGE
# =========================
print("\n--- Stage 2: Climbing image ---")

for img in images:
    img.calc = get_calc()

neb = NEB(images, k=0.2, climb=True)

opt = MDMin(neb, trajectory="neb_stage2.traj", logfile="neb_stage2.log")
opt.run(fmax=0.05)

# =========================
# FINAL ENERGY CALCULATION
# =========================
print("\n--- Final energies ---")

for img in images:
    img.calc = get_calc()

energies = [img.get_potential_energy() for img in images]

E0 = energies[0]
barrier = max(energies) - E0

# =========================
# PRINT RESULTS
# =========================
print("\n===== FINAL NEB RESULTS =====")

for i, e in enumerate(energies):
    print(f"Image {i}: {e:.6f} eV")

print(f"\n🔥 Migration barrier: {barrier:.6f} eV")

# =========================
# SAVE OUTPUT
# =========================
write("neb_band.xyz", images)

print("\nSaved:")
print("- neb_stage1.traj")
print("- neb_stage2.traj")
print("- neb_band.xyz")

In [10]:
from ase.io import read
from ase.mep import NEB
from ase.optimize import FIRE
from mace.calculators import MACECalculator
import torch

# =========================
# CONFIG
# =========================
model_path = "/home/mehuldarak/athena/mace_fps_training/mace_fps_split17_compiled.model"
initial_path = "initial.cif"
final_path = "final.cif"

n_images = 7
fmax = 0.05

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD STRUCTURES
# =========================
initial = read(initial_path)
final   = read(final_path)

# =========================
# IMPORTANT: REMOVE CONSTRAINTS
# =========================
initial.set_constraint(None)
final.set_constraint(None)

# =========================
# CALCULATOR
# =========================
calc = MACECalculator(
    model_paths=[model_path],
    device=device,
    default_dtype="float32"
)

# =========================
# CREATE IMAGES
# =========================
images = [initial]

for _ in range(n_images - 2):
    img = initial.copy()
    img.set_constraint(None)
    images.append(img)

images.append(final)

# attach calculator
for img in images:
    img.calc = calc

# =========================
# NEB SETUP
# =========================
neb = NEB(images)

# 🔥 IMPORTANT FIX
neb.interpolate(method="idpp", apply_constraint=False)

# =========================
# OPTIMIZE
# =========================
opt = FIRE(neb, trajectory="neb.traj", logfile="neb.log")
opt.run(fmax=fmax)

# =========================
# RESULTS
# =========================
energies = [img.get_potential_energy() for img in images]

E0 = energies[0]
barrier = max(energies) - E0

print("\n===== NEB RESULTS =====")
for i, e in enumerate(energies):
    print(f"Image {i}: {e:.6f} eV")

print(f"\nMigration barrier: {barrier:.6f} eV")

FileNotFoundError: [Errno 2] No such file or directory: 'initial.cif'